# Temporal Memory Agent with Local Point-in-Time Recall

AI agents often overwrite a fact when the world changes. That makes today's answer correct, but it makes questions such as *What did the agent know on Sunday?* impossible to answer honestly.

This tutorial builds a small, provider-neutral agent memory tool that:

- runs locally with SQLite and no API key;
- stores facts using their real event time;
- separates current recall from point-in-time recall; and
- returns a receipt whose hash can be verified independently.


## Why a timeline matters

Suppose a support agent learns on Friday that an order will ship Friday. On Sunday, the estimate changes to Monday. A normal vector store may retrieve both statements or whichever text is most similar. A temporal memory should answer **Monday now** and **Friday as of Sunday morning**, without leaking the later correction backward in time.


## Architecture

![Temporal memory agent architecture](../images/temporal-memory-agent.svg)

The agent can be powered by any model or orchestration framework. Memory is a separate tool: the agent writes facts and asks for either the current state or the state at a supplied timestamp.


## Install the local SDK

The local extra includes the SQLite engine. This tutorial explicitly selects Lians' deterministic local embedding provider so the example is reproducible and does not download a model. Use the default sentence-transformers provider for real semantic retrieval.


In [ ]:
%pip install "lians-sdk[local]==0.5.0" "nest-asyncio>=1.6" -q

## Import the small set of dependencies


In [ ]:
from __future__ import annotations

import hashlib
import json
import tempfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import nest_asyncio
from lians import LocalLiansClient

nest_asyncio.apply()

## Create an isolated local memory

A file-backed database makes memory durable across agent sessions. A temporary directory keeps this tutorial repeatable. In an application, replace the path with a stable location such as `~/.my-agent/memory.db`.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
memory = LocalLiansClient(
    db_path=Path(temp_dir.name) / "agent-memory.db",
    embedding_provider="local",
)

AGENT_ID = "shipping-support-agent"
FACT_FILTER = {"entity": "order-1842", "field": "shipping_estimate"}
QUERY = "When will order 1842 ship?"
SUNDAY_MORNING = datetime(2026, 8, 2, 12, tzinfo=UTC)

## Store the correction as a timeline

The Monday correction is deliberately inserted first. Correct recall must depend on `event_time`, not insertion order. The shared entity and field metadata tell the memory engine that both records describe successive values of the same fact.


In [ ]:
memory.add(
    agent_id=AGENT_ID,
    content="Order 1842 shipping estimate changed to Monday",
    event_time=datetime(2026, 8, 2, 15, tzinfo=UTC),
    metadata=FACT_FILTER,
    source="synthetic-order-event",
)
memory.add(
    agent_id=AGENT_ID,
    content="Order 1842 shipping estimate is Friday",
    event_time=datetime(2026, 8, 1, 9, tzinfo=UTC),
    metadata=FACT_FILTER,
    source="synthetic-order-event",
)

## Wrap recall as an agent tool

The class below is intentionally framework-neutral. Register `answer` as a tool in LangGraph, PydanticAI, CrewAI, or another agent framework, or expose the same memory through Lians' MCP server.


In [ ]:
class TemporalMemoryAgent:
    """Answer questions from either current or point-in-time memory."""

    def __init__(self, client: LocalLiansClient, agent_id: str) -> None:
        self.client = client
        self.agent_id = agent_id

    def recall(self, query: str, as_of: datetime | None = None) -> dict[str, Any]:
        """Return the full memory response, including its receipt."""
        return self.client.recall(
            agent_id=self.agent_id,
            query=query,
            filters=FACT_FILTER,
            as_of=as_of,
            k=3,
        )

    def answer(self, query: str, as_of: datetime | None = None) -> str:
        """Return the best human-readable memory for an agent prompt."""
        result = self.recall(query, as_of=as_of)
        memories = result["memories"]
        return memories[0]["content"] if memories else "No matching memory."


agent = TemporalMemoryAgent(memory, AGENT_ID)

## Ask for the current state


In [ ]:
current_answer = agent.answer(QUERY)
print(current_answer)
assert "Monday" in current_answer

## Ask what was true before the correction

The cutoff is Sunday at noon, three hours before the Monday estimate arrived.


In [ ]:
historical_answer = agent.answer(QUERY, as_of=SUNDAY_MORNING)
print(historical_answer)
assert "Friday" in historical_answer
assert "Monday" not in historical_answer

## Verify the recall receipts

Each recall includes the retrieval inputs and selected records in a receipt plus a SHA-256 digest. This makes it possible to detect accidental mutation between retrieval and downstream use.


In [ ]:
def verify_receipt(result: dict[str, Any]) -> bool:
    """Recompute the canonical receipt digest."""
    payload = json.dumps(
        result["receipt"],
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode()
    return hashlib.sha256(payload).hexdigest() == result["receipt_sha256"]


current_result = agent.recall(QUERY)
historical_result = agent.recall(QUERY, as_of=SUNDAY_MORNING)
assert verify_receipt(current_result)
assert verify_receipt(historical_result)
print("PASS: current and point-in-time memory stayed separated")

## Comparison

| Approach | Current answer | Historical answer | Correction semantics | Verifiable retrieval |
|---|---:|---:|---:|---:|
| Chat history | Yes | No | Position-based | No |
| Basic vector store | Usually | No | Similarity-based | Usually no |
| Temporal memory tool | Yes | Yes | Event-time + fact identity | Yes |

This example isolates temporal behavior by using deterministic test-grade embeddings. For production, use a semantic embedding provider and design stable entity/field metadata for each fact family. Point-in-time recall also depends on receiving trustworthy event timestamps from source systems.


## Cleanup and next steps

In a real agent, keep the client open for the application lifetime and use a persistent database path. The same memory can be connected to desktop agents through MCP or called directly from Python and TypeScript SDKs.


In [ ]:
memory.close()
temp_dir.cleanup()

## References

- [Lians source code](https://github.com/Lians-ai/Lians)
- [Lians documentation](https://www.lians.ai/docs)
- [Model Context Protocol](https://modelcontextprotocol.io/)
